In [ ]:
!pip install pillow
!pip install matplotlib
!pip install datasets pillow

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from datasets import load_dataset
import numpy as np

# load the image
# image = Image.open("cat.jpg")
def preprocess_image(image):
    image = image.convert("RGB")
    image = image.resize((128, 128))
    image = np.array(image).astype(np.float32) / 255.0
    return image

# Load dataset in streaming mode
dataset = load_dataset(
    "microsoft/cats_vs_dogs",
    split="train",
    streaming=True
)

images = []
labels = []

for sample in dataset.take(32):
  images.append(preprocess_image(sample["image"]))
  labels.append(sample["labels"])

X = np.array(images) #shape (32,128,128,3)
y = np.array(labels) #shape (32,)

#fliters
filters = np.random.randn(64, 27) * 0.05
filters_2 = np.random.randn(32, 576) * 0.05
filters_3 = np.random.randn(16, 288) * 0.05

b = 0.0
stride = 1
pad = 0
all_feature_maps = []

for image in X[:1]:
  #1st layer
  feature_maps_1 = []
  feature_maps_2 = []
  feature_maps_3 = []
  feature_maps_3_grids = []

  for w in filters:
    feature_map = np.zeros((126, 126))

    for i in range(126):
        for j in range(126):
            a = image[i:i+3, j:j+3].flatten()
            z = np.dot(w, a) + b
            feature_map[i, j] = z
    feature_map = np.maximum(feature_map, 0) #ReLU
    feature_maps_1.append(feature_map)

  feature_maps_1 = np.array(feature_maps_1)

  #2nd layer
  for w in filters_2:
    feature_map = np.zeros((124, 124))
    for i in range(124):
      for j in range(124):
        a = feature_maps_1[:,i:i+3, j:j+3].flatten()
        z = np.dot(w, a) + b
        feature_map[i, j] = z
    feature_map = np.maximum(feature_map, 0) #ReLU
    feature_maps_2.append(feature_map)

  feature_maps_2 = np.array(feature_maps_2)

  #3rd layer
  for w in filters_3:
    feature_map = np.zeros((122, 122))
    for i in range(122):
      for j in range(122):
        a = feature_maps_2[:,i:i+3, j:j+3].flatten()
        z = np.dot(w, a) + b
        feature_map[i, j] = z
    feature_map = np.maximum(feature_map, 0) #ReLU
    feature_maps_3_grids.append(feature_map)
    feature_map = np.mean(feature_map) #global average pooling
    feature_maps_3.append(feature_map)
  all_feature_maps.append(feature_maps_3)

all_feature_maps = np.array(all_feature_maps)
feature_maps_3_grids = np.array(feature_maps_3_grids)

dense_weights = np.random.randn(16,2)* 0.01
dense_bias = np.zeros(2)

logits = all_feature_maps @ dense_weights + dense_bias
exp_values = np.exp(logits)
p = exp_values / np.sum(exp_values,axis=1,keepdims=True)
loss = -np.log(p[0, y[0]])
print(loss)
print(p)
d_logits = p.copy()
d_logits[0, y[0]] -= 1

grad_d_weights = all_feature_maps.T @ d_logits
grad_d_bias = np.sum(d_logits, axis=0)
d_all_feature_maps = d_logits @ dense_weights.T

print(d_all_feature_maps.shape)

d_feature_maps_3 = np.ones((16, 122, 122)) * (
    d_all_feature_maps[0][:, None, None] / (122 * 122)
)

d_feature_maps_3 *= feature_maps_3_grids > 0

grad_filters_3 = np.zeros_like(filters_3)
d_feature_maps_2 = np.zeros_like(feature_maps_2)

for f in range(16):
  for i in range(122):
    for j in range(122):
      grad = d_feature_maps_3[f,i,j]
      if grad == 0:
        continue
      a = feature_maps_2[:,i:i+3, j:j+3].flatten()
      grad_filters_3[f] += grad * a
      d_feature_maps_2[:,i:i+3, j:j+3] += (grad * filters_3[f]).reshape(32,3,3)

d_feature_maps_2 *= feature_maps_2 > 0

grad_filters_2 = np.zeros_like(filters_2)
d_feature_maps_1 = np.zeros_like(feature_maps_1)

for f in range(32):
  for i in range(124):
    for j in range(124):
      grad = d_feature_maps_2[f,i,j]
      if grad == 0:
        continue
      a = feature_maps_1[:,i:i+3,j:j+3].flatten()
      grad_filters_2[f] += grad*a
      d_feature_maps_1[:,i:i+3,j:j+3] += (grad * filters_2[f]).reshape(64,3,3)

d_feature_maps_1 *= feature_maps_1 > 0
grad_filters_1 = np.zeros_like(filters)

for f in range(64):
  for i in range(126):
    for j in range(126):
      grad = d_feature_maps_1[f,i,j]
      if grad == 0:
        continue
      a = image[i:i+3,j:j+3].flatten()
      grad_filters_1[f] += grad*a



lr = 0.01
dense_weights -= lr * grad_d_weights
dense_bias -= lr * grad_d_bias
filters_3 -= lr * grad_filters_3
filters_2 -= lr * grad_filters_2
filters -= lr * grad_filters_1

Epoch 00 | Loss: 0.6934 | Probs: [0.4999 0.5001] | Target: 0
Epoch 01 | Loss: 0.6679 | Probs: [0.5128 0.4872] | Target: 0
Epoch 02 | Loss: 0.6437 | Probs: [0.5253 0.4747] | Target: 0
Epoch 03 | Loss: 0.6207 | Probs: [0.5375 0.4625] | Target: 0
Epoch 04 | Loss: 0.5989 | Probs: [0.5494 0.4506] | Target: 0
Epoch 05 | Loss: 0.5780 | Probs: [0.561 0.439] | Target: 0
Epoch 06 | Loss: 0.5581 | Probs: [0.5723 0.4277] | Target: 0
Epoch 07 | Loss: 0.5392 | Probs: [0.5832 0.4168] | Target: 0
Epoch 08 | Loss: 0.5211 | Probs: [0.5939 0.4061] | Target: 0
Epoch 09 | Loss: 0.5037 | Probs: [0.6043 0.3957] | Target: 0
Epoch 10 | Loss: 0.4871 | Probs: [0.6144 0.3856] | Target: 0
Epoch 11 | Loss: 0.4711 | Probs: [0.6243 0.3757] | Target: 0
Epoch 12 | Loss: 0.4558 | Probs: [0.6339 0.3661] | Target: 0
Epoch 13 | Loss: 0.4410 | Probs: [0.6434 0.3566] | Target: 0
Epoch 14 | Loss: 0.4268 | Probs: [0.6526 0.3474] | Target: 0
Epoch 15 | Loss: 0.4130 | Probs: [0.6617 0.3383] | Target: 0
Epoch 16 | Loss: 0.3996 | 